# Derived Tables (Subqueries in FROM)

## Overview
A **derived table** is a subquery placed in the `FROM` clause. The database executes it first and treats the result as a temporary table for the rest of the query.

```sql
SELECT outer_col, aggregated_col
FROM (
    SELECT col1, col2, computed_expression
    FROM   base_table
    WHERE  condition
) AS derived_name          -- alias is required in most databases
WHERE outer_condition;
```

**Key properties:**
- Must be aliased (required in PostgreSQL; SQLite allows omission but aliasing is good practice)
- Can contain any valid SELECT including aggregates, window functions, joins
- Invisible outside the containing query — not reusable like a CTE
- Evaluated before the outer query sees any rows

**When derived tables are the right tool:**
- Filtering on an aggregate result (can't use WHERE on aggregates directly)
- Applying a window function and then filtering on its result
- Pre-aggregating before joining to avoid row fan-out
- Breaking a complex query into two logical layers

**Derived table vs CTE:** Both achieve the same result. CTEs (`WITH` clause) are more readable for complex multi-step logic and can be referenced multiple times. Derived tables are more compact for simple two-layer queries and are supported in all SQL databases including very old ones.

---

In [ ]:
import sqlite3
import pandas as pd

conn = sqlite3.connect(':memory:')
conn.executescript("""
CREATE TABLE patients (patient_id INTEGER PRIMARY KEY, first_name TEXT, last_name TEXT, dob TEXT, sex TEXT, province TEXT, active INTEGER DEFAULT 1);
CREATE TABLE providers (provider_id INTEGER PRIMARY KEY, full_name TEXT, specialty TEXT, province TEXT, manager_id INTEGER);
CREATE TABLE departments (dept_id INTEGER PRIMARY KEY, dept_name TEXT, building TEXT, head_id INTEGER);
CREATE TABLE encounters (enc_id INTEGER PRIMARY KEY, patient_id INTEGER, provider_id INTEGER, dept_id INTEGER, enc_date TEXT, diag_code TEXT, cost_cad REAL, admitted INTEGER);
CREATE TABLE diagnoses (diag_code TEXT PRIMARY KEY, description TEXT, category TEXT);
CREATE TABLE lab_results (result_id INTEGER PRIMARY KEY, patient_id INTEGER, test_name TEXT, result_val REAL, units TEXT, collected TEXT, flagged INTEGER);
INSERT INTO patients VALUES
  (1,'Aroha','Ngata','1985-03-12','F','NB',1),(2,'Liam','Chen','1972-11-04','M','NS',1),
  (3,'Fatima','Al-Rashid','1990-07-22','F','ON',1),(4,'James','MacLeod','1955-01-30','M','NB',0),
  (5,'Sofia','Petrov','2001-09-15','F','BC',1),(6,'Noah','Williams','1968-06-08','M','AB',1),
  (7,'Mei','Zhang','1995-04-17','F','ON',1),(8,'Ethan','Tremblay','1980-12-01','M','QC',0),
  (9,'Priya','Nair','1977-08-25','F','BC',1),(10,'Marcus','Okafor','1993-05-19','M','ON',1),
  (11,'Diana','Flores','2000-02-14','F','NB',1);
INSERT INTO providers VALUES
  (10,'Dr. Sarah MacNeil','Cardiology','NB',NULL),(11,'Dr. James Wong','Oncology','BC',10),
  (12,'Dr. Fatima Osei','Family Medicine','ON',10),(13,'Dr. Ethan Larson','Orthopaedics','QC',10),
  (14,'Dr. Priya Sharma','Emergency','AB',10),(15,'Dr. Lucas Petit','Radiology','QC',11);
INSERT INTO departments VALUES
  (1,'Emergency','Tower A',14),(2,'Cardiology','Tower B',10),(3,'Oncology','Tower C',11),
  (4,'Family Medicine','Clinic',12),(5,'Orthopaedics','Tower A',13),
  (6,'Radiology','Tower B',15),(7,'ICU','Tower C',NULL);
INSERT INTO encounters VALUES
  (1,1,10,2,'2023-01-15','I25.1',3200.00,1),(2,2,14,1,'2023-02-20','J18.9',1850.00,1),
  (3,3,12,4,'2023-03-05','Z00.0',120.00,0),(4,4,13,5,'2023-03-18','M17.1',5500.00,1),
  (5,5,12,4,'2023-04-02','J06.9',95.00,0),(6,6,14,1,'2023-05-11','R07.9',780.00,0),
  (7,7,10,2,'2023-06-22','I10',2100.00,1),(8,8,12,4,'2023-07-14',NULL,80.00,0),
  (9,1,14,1,'2023-08-30','R55',410.00,0),(10,3,12,4,'2023-09-12','Z00.0',110.00,0),
  (11,9,10,2,'2023-10-01','I48.0',1750.00,1),(12,10,14,1,'2023-11-03','S52.5',2200.00,1),
  (13,2,10,2,'2023-11-20','I25.1',2900.00,1),(14,6,12,4,'2023-12-01','Z00.0',115.00,0);
INSERT INTO diagnoses VALUES
  ('I25.1','Atherosclerotic heart disease','Cardiovascular'),('J18.9','Pneumonia, unspecified','Respiratory'),
  ('Z00.0','General medical examination','Preventive'),('M17.1','Primary osteoarthritis of knee','Musculoskeletal'),
  ('J06.9','Acute upper respiratory infection','Respiratory'),('R07.9','Chest pain, unspecified','Symptoms'),
  ('I10','Essential hypertension','Cardiovascular'),('R55','Syncope and collapse','Symptoms'),
  ('I48.0','Paroxysmal atrial fibrillation','Cardiovascular'),('S52.5','Fracture of lower end of radius','Injury'),
  ('E11.9','Type 2 diabetes mellitus','Endocrine'),('I50.9','Heart failure, unspecified','Cardiovascular');
INSERT INTO lab_results VALUES
  (1,1,'HbA1c',7.2,'%','2023-03-10',0),(2,2,'HbA1c',9.1,'%','2023-03-15',1),
  (3,3,'Creatinine',88.5,'umol/L','2023-04-01',0),(4,4,'Creatinine',145.0,'umol/L','2023-04-12',1),
  (5,5,'HbA1c',5.8,'%','2023-05-05',0),(6,6,'Sodium',138.0,'mmol/L','2023-05-20',0),
  (7,7,'Sodium',151.0,'mmol/L','2023-06-01',1),(8,1,'Creatinine',72.3,'umol/L','2023-06-18',0),
  (9,8,'HbA1c',6.4,'%','2023-07-02',0),(10,9,'Creatinine',210.0,'umol/L','2023-07-15',1),
  (11,2,'Creatinine',95.0,'umol/L','2023-08-01',0),(12,10,'HbA1c',7.8,'%','2023-08-20',1);
""")
conn.commit()
print('Healthcare schema ready.')


---
## Basic derived table — filtering on an aggregate

In [ ]:
# Cannot use WHERE on an aggregate — must wrap in a derived table (or use HAVING)
# Example: find patients whose total encounter cost exceeds $3000
print('=== Patients with total encounter cost > $3000 (derived table) ===')
q = """
SELECT  patient_id,
        total_cost,
        encounter_count
FROM (
    SELECT  patient_id,
            ROUND(SUM(cost_cad), 2)  AS total_cost,
            COUNT(*)                 AS encounter_count
    FROM    encounters
    GROUP BY patient_id
) AS patient_costs
WHERE   total_cost > 3000
ORDER BY total_cost DESC
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Could also use HAVING here — derived table is useful when the outer query')
print('needs to join or further filter the aggregated result.')


---
## Filtering on a window function result

In [ ]:
# Window functions can't appear in WHERE — must wrap in derived table or CTE
print('=== Top 2 encounters by cost per department (window function + derived table) ===')
q = """
SELECT  enc_id, dept_id, enc_date, diag_code, cost_cad, rn
FROM (
    SELECT  enc_id,
            dept_id,
            enc_date,
            diag_code,
            cost_cad,
            ROW_NUMBER() OVER (
                PARTITION BY dept_id
                ORDER BY cost_cad DESC, enc_id
            ) AS rn
    FROM    encounters
) AS ranked
WHERE   rn <= 2
ORDER BY dept_id, rn
"""
print(pd.read_sql(q, conn).to_string(index=False))


---
## Pre-aggregating before a join — preventing row fan-out

In [ ]:
# Bad approach: join then aggregate — if a patient has multiple labs,
# encounter rows are duplicated before the GROUP BY
print('=== WRONG: join then aggregate creates fan-out ===')
q_bad = """
SELECT  e.patient_id,
        COUNT(DISTINCT e.enc_id)   AS encounters,
        COUNT(lr.result_id)        AS lab_results,
        ROUND(SUM(e.cost_cad), 2)  AS total_cost
FROM    encounters  AS e
LEFT JOIN lab_results AS lr ON e.patient_id = lr.patient_id
GROUP BY e.patient_id
ORDER BY e.patient_id
"""
print('Join-then-aggregate (encounter costs may be duplicated per lab row):')
print(pd.read_sql(q_bad, conn).to_string(index=False))

# Good approach: pre-aggregate each side, then join
print()
print('=== CORRECT: pre-aggregate each side, then join ===')
q_good = """
SELECT  ec.patient_id,
        ec.encounter_count,
        ec.total_cost,
        COALESCE(lr.lab_count, 0)   AS lab_results
FROM (
    SELECT  patient_id,
            COUNT(*)                AS encounter_count,
            ROUND(SUM(cost_cad), 2) AS total_cost
    FROM    encounters
    GROUP BY patient_id
) AS ec
LEFT JOIN (
    SELECT  patient_id,
            COUNT(*)   AS lab_count
    FROM    lab_results
    GROUP BY patient_id
) AS lr ON ec.patient_id = lr.patient_id
ORDER BY ec.patient_id
"""
print('Pre-aggregate then join (each side independently summarised):')
print(pd.read_sql(q_good, conn).to_string(index=False))


---
## Multi-layer derived tables

In [ ]:
# Three-layer query: base → aggregate → classify
print('=== Three-layer derived table: encounter data → cost bucket → summary ===')
q = """
SELECT  cost_tier,
        COUNT(*)                    AS patients,
        ROUND(AVG(total_cost), 2)   AS avg_cost_in_tier,
        ROUND(MIN(total_cost), 2)   AS min_cost,
        ROUND(MAX(total_cost), 2)   AS max_cost
FROM (
    -- Layer 2: classify each patient's total cost into a tier
    SELECT  patient_id,
            total_cost,
            CASE
                WHEN total_cost < 500    THEN 'Low (<$500)'
                WHEN total_cost < 2000   THEN 'Medium ($500-$2k)'
                WHEN total_cost < 5000   THEN 'High ($2k-$5k)'
                ELSE                          'Very High (>$5k)'
            END AS cost_tier
    FROM (
        -- Layer 1: total cost per patient
        SELECT  patient_id,
                ROUND(SUM(cost_cad), 2) AS total_cost
        FROM    encounters
        GROUP BY patient_id
    ) AS patient_totals
) AS tiered
GROUP BY cost_tier
ORDER BY MIN(total_cost)
"""
print(pd.read_sql(q, conn).to_string(index=False))
print()
print('Equivalent CTE version (preferred for readability — see basic_ctes notebook):')
print("""
WITH patient_totals AS (
    SELECT patient_id, ROUND(SUM(cost_cad),2) AS total_cost
    FROM encounters GROUP BY patient_id
),
tiered AS (
    SELECT patient_id, total_cost,
           CASE WHEN total_cost < 500  THEN 'Low'
                WHEN total_cost < 2000 THEN 'Medium'
                ELSE 'High' END AS cost_tier
    FROM patient_totals
)
SELECT cost_tier, COUNT(*), AVG(total_cost) FROM tiered GROUP BY cost_tier;
""")

conn.close()


---
## Common Pitfalls

**1. Forgetting to alias the derived table**
PostgreSQL requires every derived table to have an alias — `FROM (...) AS dt`. SQLite is more lenient but aliasing is essential for readability and portability. Always name derived tables descriptively: `patient_costs`, `ranked_encounters`.

**2. Join-then-aggregate fan-out producing inflated totals**
Joining encounters to lab_results before aggregating means each encounter row is duplicated once per matching lab result. `SUM(cost_cad)` then counts those costs multiple times. Pre-aggregate each side independently, then join the summaries.

**3. Derived table columns can't be referenced by alias in WHERE of the same layer**
`FROM (...) AS dt WHERE dt.total_cost > 1000` — the column `total_cost` defined inside the subquery is accessible via `dt.total_cost` or just `total_cost` in the outer query. But inside the derived table itself, you can't reference its own column aliases in its WHERE clause — that's what the outer layer is for.

**4. Deep nesting becomes unreadable quickly**
Three levels of nested derived tables are hard to read and debug. Beyond two layers, switch to CTEs — they give each intermediate step a name and allow easy inspection of intermediate results during development.

**5. Derived tables are not reusable within the same query**
If you need the same derived result in multiple places (e.g. to JOIN and also to compute a percentage), you must either repeat the subquery or use a CTE. Duplicating complex subqueries is a maintenance hazard.

**6. ORDER BY inside a derived table is ignored in most databases**
`FROM (SELECT ... ORDER BY enc_date) AS dt` — the ORDER BY inside the subquery has no guaranteed effect on the order of rows seen by the outer query. PostgreSQL ignores it; SQLite may or may not preserve it. Always put ORDER BY in the outermost query.


---
*sql_methods_library - Samantha McGarrigle*